In [1]:
!pip install cantera
!pip install CoolProp
import cantera as ct
print(ct.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.0 MB/s eta 0:00:00
3.2.0


In [2]:
# pip install CoolProp pandas matplotlib
# Optional for Excel export: pip install openpyxl

import os
import pandas as pd
from CoolProp.CoolProp import PropsSI

FLUID = "Methane"
REQUIRED_PACKAGES = ["CoolProp", "pandas"]


def C_to_K(T_C):
    return T_C + 273.15


def K_to_C(T_K):
    return T_K - 273.15


def bar_to_Pa(P_bar):
    return P_bar * 1e5


def Pa_to_bar(P_Pa):
    return P_Pa / 1e5


# =====================================================
# 1. CoolProp property helpers
# =====================================================

def h_TP(T, P, fluid=FLUID):
    try:
        return PropsSI("Hmass", "T", T, "P", P, fluid)
    except ValueError:
        return PropsSI("Hmass", "T", T + 0.1, "P", P, fluid)


def s_TP(T, P, fluid=FLUID):
    try:
        return PropsSI("Smass", "T", T, "P", P, fluid)
    except ValueError:
        return PropsSI("Smass", "T", T + 0.1, "P", P, fluid)


def T_Ph(P, h, fluid=FLUID):
    return PropsSI("T", "P", P, "Hmass", h, fluid)


def h_PS(P, s, fluid=FLUID):
    return PropsSI("Hmass", "P", P, "Smass", s, fluid)


def rho_TP(T, P, fluid=FLUID):
    try:
        return PropsSI("Dmass", "T", T, "P", P, fluid)
    except ValueError:
        return PropsSI("Dmass", "T", T + 0.1, "P", P, fluid)


# =====================================================
# 2. Component models
# =====================================================

def liquid_pump(m_dot, T_in, P_in, P_out, eta=0.75):
    h_in = h_TP(T_in, P_in)
    rho = rho_TP(T_in, P_in)

    if rho < 100.0:
        raise ValueError(
            f"Liquid pump inlet density is too low ({rho:.3f} kg/m3). "
            "The inlet is likely vapor/two-phase, not liquid. "
            "Increase pump inlet pressure or lower pump inlet temperature."
        )

    dh = (P_out - P_in) / rho / eta
    h_out = h_in + dh
    T_out = T_Ph(P_out, h_out)
    W = m_dot * dh

    return {
        "T_out": T_out,
        "P_out": P_out,
        "h_out": h_out,
        "W": W,
        "rho_in": rho,
    }


def compressor(m_dot, T_in, P_in, P_out, eta=0.75):
    if m_dot <= 0:
        return {
            "T_out": T_in,
            "P_out": P_out,
            "h_out": h_TP(T_in, P_out),
            "W": 0.0,
        }

    h_in = h_TP(T_in, P_in)
    s_in = s_TP(T_in, P_in)
    h_out_s = h_PS(P_out, s_in)
    h_out = h_in + (h_out_s - h_in) / eta
    T_out = T_Ph(P_out, h_out)
    W = m_dot * (h_out - h_in)

    return {
        "T_out": T_out,
        "P_out": P_out,
        "h_out": h_out,
        "W": W,
    }


def heat_exchanger_effectiveness(
    m_cold,
    T_cold_in,
    P_cold,
    m_hot,
    T_hot_in,
    P_hot,
    effectiveness=0.60,
    min_delta_T=3.0,
):
    """
    Generic hot-to-cold heat exchanger effectiveness model.
    Hot stream gives heat to cold stream.
    """
    if m_hot <= 0 or effectiveness <= 0:
        return {
            "Q": 0.0,
            "T_cold_out": T_cold_in,
            "h_cold_out": h_TP(T_cold_in, P_cold),
            "T_hot_out": T_hot_in,
            "h_hot_out": h_TP(T_hot_in, P_hot),
        }

    h_hot_in = h_TP(T_hot_in, P_hot)
    h_cold_in = h_TP(T_cold_in, P_cold)
    T_hot_min = T_cold_in + min_delta_T
    h_hot_min = h_TP(T_hot_min, P_hot)

    Q_max_hot = m_hot * max(h_hot_in - h_hot_min, 0.0)
    Q = effectiveness * Q_max_hot

    h_hot_out = h_hot_in - Q / m_hot
    h_cold_out = h_cold_in + Q / m_cold

    T_hot_out = T_Ph(P_hot, h_hot_out)
    T_cold_out = T_Ph(P_cold, h_cold_out)

    return {
        "Q": Q,
        "T_cold_out": T_cold_out,
        "h_cold_out": h_cold_out,
        "T_hot_out": T_hot_out,
        "h_hot_out": h_hot_out,
    }


def set_stream_temperature(m_dot, T_in, P, T_target):
    """
    Apply a heater/cooler to set a stream temperature at the same pressure.
    Positive Q means heat is added to the stream.
    """
    h_in = h_TP(T_in, P)
    h_out = h_TP(T_target, P)
    Q = m_dot * (h_out - h_in)

    return {
        "Q": Q,
        "T_out": T_target,
        "P_out": P,
        "h_out": h_out,
    }


def mix_streams_same_pressure(streams, P_mix):
    """
    Mix multiple streams at the same target pressure using enthalpy balance.
    Each stream dict must have m_dot, T, P.

    For this simplified model, pressure matching to P_mix is treated as isenthalpic
    if stream pressure differs from P_mix. This is a modeling assumption for the
    reliquefaction return line.
    """
    total_m = sum(s["m_dot"] for s in streams)
    if total_m <= 0:
        raise ValueError("Total mixed mass flow must be positive.")

    total_h_flow = 0.0
    for s in streams:
        h = h_TP(s["T"], s["P"])
        total_h_flow += s["m_dot"] * h

    h_mix = total_h_flow / total_m
    T_mix = T_Ph(P_mix, h_mix)

    return {
        "m_dot": total_m,
        "T": T_mix,
        "P": P_mix,
        "h": h_mix,
    }


def vaporizer_for_mixed_sendout(
    m_ng,
    T_ng_in,
    P_ng_in,
    m_bog,
    T_bog_to_mixer,
    P_bog_to_mixer,
    T_mix_target,
    P_mix,
):
    """
    Enthalpy mixing model.
    Vaporizer outlet NG + BOG after throttling must make final mixed gas at target T/P.
    """
    h_ng_in = h_TP(T_ng_in, P_ng_in)

    h_bog_before_valve = h_TP(T_bog_to_mixer, P_bog_to_mixer)
    h_bog_mix = h_bog_before_valve
    T_bog_after_valve = T_Ph(P_mix, h_bog_mix)

    h_mix_target = h_TP(T_mix_target, P_mix)
    h_ng_vap_out = ((m_ng + m_bog) * h_mix_target - m_bog * h_bog_mix) / m_ng
    T_ng_vap_out = T_Ph(P_mix, h_ng_vap_out)
    Q = m_ng * (h_ng_vap_out - h_ng_in)

    return {
        "Q": Q,
        "T_ng_vap_out": T_ng_vap_out,
        "h_ng_vap_out": h_ng_vap_out,
        "T_bog_after_valve": T_bog_after_valve,
        "T_mix_out": T_mix_target,
        "P_mix_out": P_mix,
    }


# =====================================================
# 3. Multi-stage HPC model
# =====================================================

def get_hpc_stage_pressures(P_in, P_out, n_stages):
    """
    Equal pressure ratio stages.
    Returns boundaries: [P0, P1, ..., Pn]
    """
    pr_stage = (P_out / P_in) ** (1.0 / n_stages)
    pressures = [P_in]
    for _ in range(n_stages):
        pressures.append(pressures[-1] * pr_stage)
    pressures[-1] = P_out
    return pressures


def multistage_hpc_no_intercooling(m_dot, T_in, P_in, P_out, eta=0.75, n_stages=3):
    pressures = get_hpc_stage_pressures(P_in, P_out, n_stages)
    T = T_in
    W_total = 0.0
    stage_results = []

    for i in range(n_stages):
        T_stage_in = T
        comp = compressor(m_dot, T_stage_in, pressures[i], pressures[i + 1], eta)
        W_total += comp["W"]
        T = comp["T_out"]

        stage_results.append({
            "stage": i + 1,
            "P_in": pressures[i],
            "P_out": pressures[i + 1],
            "T_in": T_stage_in,
            "T_out": comp["T_out"],
            "W": comp["W"],
        })

    return {
        "T_out": T,
        "P_out": P_out,
        "W": W_total,
        "stages": stage_results,
    }


def multistage_hpc_with_lng_intercooling(
    m_bog,
    T_bog_in,
    P_bog_in,
    P_bog_out,
    m_cold,
    T_cold_in,
    P_cold,
    eta=0.75,
    n_stages=3,
    intercooler_eff=0.50,
    min_delta_T=10.0,
):
    """
    3-stage high-pressure compressor with cold NG/LNG stream heat exchangers
    before stage 2 and stage 3.

    The cold stream is the main vaporizer feed after pump-out heating and
    reliquefaction branch return mixing.
    """
    pressures = get_hpc_stage_pressures(P_bog_in, P_bog_out, n_stages)
    T_bog = T_bog_in
    T_cold = T_cold_in
    W_total = 0.0
    Q_intercool_total = 0.0
    stage_results = []
    hx_results = []

    for i in range(n_stages):
        T_stage_in = T_bog
        P_stage_in = pressures[i]
        P_stage_out = pressures[i + 1]

        comp = compressor(m_bog, T_stage_in, P_stage_in, P_stage_out, eta)
        W_total += comp["W"]
        T_bog = comp["T_out"]

        stage_results.append({
            "stage": i + 1,
            "P_in": P_stage_in,
            "P_out": P_stage_out,
            "T_in": T_stage_in,
            "T_out": T_bog,
            "W": comp["W"],
        })

        if i < n_stages - 1:
            hx = heat_exchanger_effectiveness(
                m_cold=m_cold,
                T_cold_in=T_cold,
                P_cold=P_cold,
                m_hot=m_bog,
                T_hot_in=T_bog,
                P_hot=P_stage_out,
                effectiveness=intercooler_eff,
                min_delta_T=min_delta_T,
            )
            Q_intercool_total += hx["Q"]

            hx_results.append({
                "hx": i + 1,
                "after_stage": i + 1,
                "Q": hx["Q"],
                "BOG_T_in": T_bog,
                "BOG_T_out": hx["T_hot_out"],
                "BOG_P": P_stage_out,
                "COLD_T_in": T_cold,
                "COLD_T_out": hx["T_cold_out"],
                "COLD_P": P_cold,
            })

            T_bog = hx["T_hot_out"]
            T_cold = hx["T_cold_out"]

    return {
        "T_bog_out": T_bog,
        "P_bog_out": P_bog_out,
        "T_cold_out": T_cold,
        "P_cold": P_cold,
        "W": W_total,
        "Q_intercool": Q_intercool_total,
        "stages": stage_results,
        "hxs": hx_results,
    }


# =====================================================
# 4. Overall process model
# =====================================================

def run_case(params, case_name):
    m_lng = params["m_lng"]
    m_bog_total = params["m_bog_total"]
    bog_to_hpc_frac = params["bog_to_hpc_frac"]

    m_bog_hpc = m_bog_total * bog_to_hpc_frac
    m_bog_pc = m_bog_total - m_bog_hpc

    P_lng_tank = params["P_lng_tank"]
    P_lng_high = params["P_lng_high"]
    P_sendout = params["P_sendout"]

    T_lng_tank = params["T_lng_tank"]
    T_lng_pump_out_heated = params["T_lng_pump_out_heated"]

    P_bog_tank = params["P_bog_tank"]
    P_bog_comp_out = params["P_bog_comp_out"]
    P_bog_hpc_out = params["P_bog_hpc_out"]

    T_sendout = params["T_sendout"]

    pump_eta = params["pump_eta"]
    comp_eta = params["comp_eta"]
    precooler_eff = params["precooler_eff"]
    hpc_intercooler_eff = params["hpc_intercooler_eff"]

    # LNG pump: true pump inlet stays at storage-tank pressure and temperature.
    pump = liquid_pump(
        m_dot=m_lng,
        T_in=T_lng_tank,
        P_in=P_lng_tank,
        P_out=P_lng_high,
        eta=pump_eta,
    )

    T_lng_after_pump_calc = pump["T_out"]
    P_lng_after_pump = pump["P_out"]

    # Post-pump heat leak / upstream heat exchange: force high-pressure LNG to about -130 C.
    pump_out_heater = set_stream_temperature(
        m_dot=m_lng,
        T_in=T_lng_after_pump_calc,
        P=P_lng_after_pump,
        T_target=T_lng_pump_out_heated,
    )

    T_lng_after_pump_heated = pump_out_heater["T_out"]
    Q_lng_pump_out_heating = pump_out_heater["Q"]

    # BOG compressor: total BOG before split.
    comp_bog = compressor(
        m_dot=m_bog_total,
        T_in=params["T_bog_tank"],
        P_in=P_bog_tank,
        P_out=P_bog_comp_out,
        eta=comp_eta,
    )

    T_bog_comp_out = comp_bog["T_out"]
    P_bog_comp_out = comp_bog["P_out"]

    Q_precooler = 0.0
    Q_hpc_intercool = 0.0
    T_main_after_precooler = None
    T_bog_precooler_out = None
    T_after_reliq_mix = None
    T_after_hpc_intercoolers = None
    hpc_stage_results = []
    hpc_hx_results = []

    # Main high-pressure LNG stream after pump outlet heating.
    m_vaporizer_feed = m_lng
    T_vaporizer_feed = T_lng_after_pump_heated
    P_vaporizer_feed = P_lng_after_pump

    # Case 1/2/3: BOG reliquefaction return branch is included in vaporizer feed.
    # Case 1: no precooler heat recovery. The BOG_PC branch is assumed to be separately
    #         reliquefied and returned to the LNG line at a fixed return temperature.
    # Case 2/3: the BOG_PC branch first exchanges heat with LNG in the precooler,
    #           then returns to the LNG line.
    if case_name == "case1_base":
        T_bog_reliq_return = params["T_bog_reliq_return_case1"]

        reliq_mix = mix_streams_same_pressure(
            streams=[
                {"m_dot": m_lng, "T": T_lng_after_pump_heated, "P": P_lng_after_pump},
                {"m_dot": m_bog_pc, "T": T_bog_reliq_return, "P": P_lng_after_pump},
            ],
            P_mix=P_lng_after_pump,
        )

        T_bog_precooler_out = T_bog_reliq_return
        m_vaporizer_feed = reliq_mix["m_dot"]
        T_vaporizer_feed = reliq_mix["T"]
        P_vaporizer_feed = reliq_mix["P"]
        T_after_reliq_mix = reliq_mix["T"]

    elif case_name in ["case2_precooler", "case3_multistage_hpc_intercooling"]:
        precooler = heat_exchanger_effectiveness(
            m_cold=m_lng,
            T_cold_in=T_lng_after_pump_heated,
            P_cold=P_lng_after_pump,
            m_hot=m_bog_pc,
            T_hot_in=T_bog_comp_out,
            P_hot=P_bog_comp_out,
            effectiveness=precooler_eff,
            min_delta_T=params["precooler_min_delta_T"],
        )

        Q_precooler = precooler["Q"]
        T_main_after_precooler = precooler["T_cold_out"]
        T_bog_precooler_out = precooler["T_hot_out"]

        reliq_mix = mix_streams_same_pressure(
            streams=[
                {"m_dot": m_lng, "T": T_main_after_precooler, "P": P_lng_after_pump},
                {"m_dot": m_bog_pc, "T": T_bog_precooler_out, "P": P_lng_after_pump},
            ],
            P_mix=P_lng_after_pump,
        )

        m_vaporizer_feed = reliq_mix["m_dot"]
        T_vaporizer_feed = reliq_mix["T"]
        P_vaporizer_feed = reliq_mix["P"]
        T_after_reliq_mix = reliq_mix["T"]

    else:
        raise ValueError("case_name이 잘못되었습니다.")

    # HPC line
    if case_name == "case3_multistage_hpc_intercooling":
        hpc = multistage_hpc_with_lng_intercooling(
            m_bog=m_bog_hpc,
            T_bog_in=T_bog_comp_out,
            P_bog_in=P_bog_comp_out,
            P_bog_out=P_bog_hpc_out,
            m_cold=m_vaporizer_feed,
            T_cold_in=T_vaporizer_feed,
            P_cold=P_vaporizer_feed,
            eta=comp_eta,
            n_stages=3,
            intercooler_eff=hpc_intercooler_eff,
            min_delta_T=params["hpc_intercooler_min_delta_T"],
        )
        T_bog_to_mixer = hpc["T_bog_out"]
        P_bog_to_mixer = hpc["P_bog_out"]
        T_vaporizer_feed = hpc["T_cold_out"]
        Q_hpc_intercool = hpc["Q_intercool"]
        hpc_stage_results = hpc["stages"]
        hpc_hx_results = hpc["hxs"]
        W_hpc = hpc["W"]
        T_after_hpc_intercoolers = hpc["T_cold_out"]

    else:
        hpc = multistage_hpc_no_intercooling(
            m_dot=m_bog_hpc,
            T_in=T_bog_comp_out,
            P_in=P_bog_comp_out,
            P_out=P_bog_hpc_out,
            eta=comp_eta,
            n_stages=3,
        )
        T_bog_to_mixer = hpc["T_out"]
        P_bog_to_mixer = hpc["P_out"]
        W_hpc = hpc["W"]
        hpc_stage_results = hpc["stages"]

    vap = vaporizer_for_mixed_sendout(
        m_ng=m_vaporizer_feed,
        T_ng_in=T_vaporizer_feed,
        P_ng_in=P_vaporizer_feed,
        m_bog=m_bog_hpc,
        T_bog_to_mixer=T_bog_to_mixer,
        P_bog_to_mixer=P_bog_to_mixer,
        T_mix_target=T_sendout,
        P_mix=P_sendout,
    )

    total_power_mw = (pump["W"] + comp_bog["W"] + W_hpc) / 1e6
    vaporizer_duty_mw = vap["Q"] / 1e6
    pump_out_heating_duty_mw = Q_lng_pump_out_heating / 1e6

    return {
        "CASE": case_name,

        # split flow
        "BOG_TOTAL_MASS_FLOW": m_bog_total,
        "BOG_HPC_MASS_FLOW": m_bog_hpc,
        "BOG_PRECOOLER_MASS_FLOW": m_bog_pc,

        # LNG / vaporizer feed line
        "LNG_TANK_T_C": K_to_C(T_lng_tank),
        "LNG_TANK_P_BAR": Pa_to_bar(P_lng_tank),
        "LNG_PUMP_OUT_CALC_T_C": K_to_C(T_lng_after_pump_calc),
        "LNG_PUMP_OUT_P_BAR": Pa_to_bar(P_lng_after_pump),
        "LNG_PUMP_OUT_HEATED_T_C": K_to_C(T_lng_after_pump_heated),
        "LNG_PUMP_OUT_HEATING_DUTY_MW": pump_out_heating_duty_mw,
        "LNG_PRECOOLER_OUT_T_C": None if T_main_after_precooler is None else K_to_C(T_main_after_precooler),
        "BOG_RELIQ_RETURN_T_C": None if T_bog_precooler_out is None else K_to_C(T_bog_precooler_out),
        "AFTER_RELIQ_MIX_T_C": None if T_after_reliq_mix is None else K_to_C(T_after_reliq_mix),
        "AFTER_HPC_INTERCOOLERS_T_C": None if T_after_hpc_intercoolers is None else K_to_C(T_after_hpc_intercoolers),
        "VAPORIZER_FEED_MASS_FLOW": m_vaporizer_feed,
        "VAPORIZER_FEED_T_C": K_to_C(T_vaporizer_feed),
        "VAPORIZER_FEED_P_BAR": Pa_to_bar(P_vaporizer_feed),
        "VAPORIZER_OUT_T_C": K_to_C(vap["T_ng_vap_out"]),

        # BOG line
        "BOG_TANK_T_C": K_to_C(params["T_bog_tank"]),
        "BOG_TANK_P_BAR": Pa_to_bar(P_bog_tank),
        "BOG_COMP_OUT_T_C": K_to_C(T_bog_comp_out),
        "BOG_COMP_OUT_P_BAR": Pa_to_bar(P_bog_comp_out),
        "BOG_PRECOOLER_OUT_T_C": None if T_bog_precooler_out is None else K_to_C(T_bog_precooler_out),
        "BOG_HPC_OUT_T_C": K_to_C(T_bog_to_mixer),
        "BOG_HPC_OUT_P_BAR": Pa_to_bar(P_bog_to_mixer),
        "BOG_AFTER_VALVE_T_C": K_to_C(vap["T_bog_after_valve"]),
        "BOG_AFTER_VALVE_P_BAR": Pa_to_bar(P_sendout),

        # mixed sendout
        "MIXED_SENDOUT_T_C": K_to_C(vap["T_mix_out"]),
        "MIXED_SENDOUT_P_BAR": Pa_to_bar(P_sendout),
        "MIXED_SENDOUT_MASS_FLOW": m_vaporizer_feed + m_bog_hpc,

        # duties / powers
        "PRECOOLER_DUTY_MW": Q_precooler / 1e6,
        "HPC_INTERCOOLING_DUTY_MW": Q_hpc_intercool / 1e6,
        "VAPORIZER_DUTY_MW": vaporizer_duty_mw,
        "LNG_PUMP_POWER_MW": pump["W"] / 1e6,
        "BOG_COMP_POWER_MW": comp_bog["W"] / 1e6,
        "BOG_HPC_POWER_MW": W_hpc / 1e6,
        "TOTAL_POWER_MW": total_power_mw,

        # combined loads
        "VAPORIZER_PLUS_POWER_MW": vaporizer_duty_mw + total_power_mw,
        "HEATING_PLUS_VAPORIZER_PLUS_POWER_MW": pump_out_heating_duty_mw + vaporizer_duty_mw + total_power_mw,

        # detailed stage info
        "HPC_STAGE_RESULTS": hpc_stage_results,
        "HPC_HX_RESULTS": hpc_hx_results,
    }


# =====================================================
# 5. Input conditions
# =====================================================

params = {
    "m_lng": 65.0,

    # Total BOG flow, split after BOG compressor
    "m_bog_total": 35.0,
    "bog_to_hpc_frac": 0.60,

    # LNG storage tank and true pump inlet condition
    "T_lng_tank": C_to_K(-160.0),
    "P_lng_tank": bar_to_Pa(1.2),

    # LNG 2nd pump outlet is assumed to be heated after pumping.
    # This avoids the unphysical assumption of pressure rise before the pump.
    "T_lng_pump_out_heated": C_to_K(-130.0),

    # LNG 2nd pump outlet pressure
    "P_lng_high": bar_to_Pa(80.0),

    "T_bog_tank": C_to_K(-100.0),
    "P_bog_tank": bar_to_Pa(1.2),
    "P_bog_comp_out": bar_to_Pa(10.0),
    "P_bog_hpc_out": bar_to_Pa(80.0),

    "T_sendout": C_to_K(15.0),
    "P_sendout": bar_to_Pa(70.0),

    "pump_eta": 0.75,
    "comp_eta": 0.75,

    # Case 1 separate reliquefaction return condition.
    # No precooler heat recovery is credited in case1, but the BOG_PC branch still returns to the LNG line.
    "T_bog_reliq_return_case1": C_to_K(-124.0),

    # Case 2/3 reliquefaction/precooler branch
    "precooler_eff": 0.60,
    "precooler_min_delta_T": 3.0,

    # Case 3 high-pressure compressor inlet cooling by vaporizer feed stream
    "hpc_intercooler_eff": 0.50,
    "hpc_intercooler_min_delta_T": 10.0,
}


# =====================================================
# 6. Smoke tests
# =====================================================

def run_smoke_tests():
    p_test = params.copy()
    r1 = run_case(p_test, "case1_base")

    assert r1["LNG_PUMP_OUT_CALC_T_C"] < -140.0
    assert r1["LNG_PUMP_OUT_HEATED_T_C"] == -130.0
    expected_feed_flow_case1 = p_test["m_lng"] + p_test["m_bog_total"] * (1.0 - p_test["bog_to_hpc_frac"])
    assert abs(r1["VAPORIZER_FEED_MASS_FLOW"] - expected_feed_flow_case1) < 1e-9
    assert r1["MIXED_SENDOUT_T_C"] == 15.0
    assert r1["MIXED_SENDOUT_MASS_FLOW"] == p_test["m_lng"] + p_test["m_bog_total"]
    assert r1["AFTER_RELIQ_MIX_T_C"] is not None
    assert "VAPORIZER_PLUS_POWER_MW" in r1
    assert "HEATING_PLUS_VAPORIZER_PLUS_POWER_MW" in r1

    r2 = run_case(p_test, "case2_precooler")
    expected_feed_flow = p_test["m_lng"] + p_test["m_bog_total"] * (1.0 - p_test["bog_to_hpc_frac"])
    assert abs(r2["VAPORIZER_FEED_MASS_FLOW"] - expected_feed_flow) < 1e-9
    assert r2["AFTER_RELIQ_MIX_T_C"] is not None
    assert r2["MIXED_SENDOUT_MASS_FLOW"] == p_test["m_lng"] + p_test["m_bog_total"]

    r3 = run_case(p_test, "case3_multistage_hpc_intercooling")
    assert r3["BOG_HPC_POWER_MW"] > 0.0
    assert r3["HPC_INTERCOOLING_DUTY_MW"] >= 0.0
    assert r3["MIXED_SENDOUT_T_C"] == 15.0
    assert r3["MIXED_SENDOUT_MASS_FLOW"] == p_test["m_lng"] + p_test["m_bog_total"]

    # Export fallback test: should not require openpyxl.
    assert "openpyxl" not in REQUIRED_PACKAGES

    print("Smoke tests passed.")


run_smoke_tests()


# =====================================================
# 7. Run cases
# =====================================================

case_names = [
    "case1_base",
    "case2_precooler",
    "case3_multistage_hpc_intercooling",
]

cases = [run_case(params, c) for c in case_names]
df = pd.DataFrame(cases)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 320)

summary_cols = [
    "CASE",
    "BOG_TOTAL_MASS_FLOW",
    "BOG_HPC_MASS_FLOW",
    "BOG_PRECOOLER_MASS_FLOW",
    "LNG_PUMP_OUT_CALC_T_C",
    "LNG_PUMP_OUT_HEATED_T_C",
    "LNG_PUMP_OUT_HEATING_DUTY_MW",
    "LNG_PRECOOLER_OUT_T_C",
    "BOG_RELIQ_RETURN_T_C",
    "AFTER_RELIQ_MIX_T_C",
    "AFTER_HPC_INTERCOOLERS_T_C",
    "VAPORIZER_FEED_MASS_FLOW",
    "VAPORIZER_FEED_T_C",
    "VAPORIZER_OUT_T_C",
    "BOG_COMP_OUT_T_C",
    "BOG_PRECOOLER_OUT_T_C",
    "BOG_HPC_OUT_T_C",
    "BOG_AFTER_VALVE_T_C",
    "MIXED_SENDOUT_T_C",
    "MIXED_SENDOUT_MASS_FLOW",
    "PRECOOLER_DUTY_MW",
    "HPC_INTERCOOLING_DUTY_MW",
    "VAPORIZER_DUTY_MW",
    "BOG_HPC_POWER_MW",
    "TOTAL_POWER_MW",
    "VAPORIZER_PLUS_POWER_MW",
    "HEATING_PLUS_VAPORIZER_PLUS_POWER_MW",
]

print("\n=== LNG 기화 공정 3개 Case 비교 ===")
print(df[summary_cols].to_string(index=False))


# =====================================================
# 8. State tables
# =====================================================

def make_state_table(params, case_name):
    r = run_case(params, case_name)

    m_lng = params["m_lng"]
    m_bog_total = r["BOG_TOTAL_MASS_FLOW"]
    m_bog_hpc = r["BOG_HPC_MASS_FLOW"]
    m_bog_pc = r["BOG_PRECOOLER_MASS_FLOW"]

    rows = []

    rows.append(["LNG-01", "LNG tank / pump inlet", m_lng, r["LNG_TANK_T_C"], r["LNG_TANK_P_BAR"]])
    rows.append(["LNG-02", "LNG pump outlet calculated", m_lng, r["LNG_PUMP_OUT_CALC_T_C"], r["LNG_PUMP_OUT_P_BAR"]])
    rows.append(["LNG-03", "LNG pump outlet after heating", m_lng, r["LNG_PUMP_OUT_HEATED_T_C"], r["LNG_PUMP_OUT_P_BAR"]])

    if r["LNG_PRECOOLER_OUT_T_C"] is not None:
        rows.append(["LNG-04", "Main LNG outlet after BOG precooler", m_lng, r["LNG_PRECOOLER_OUT_T_C"], r["LNG_PUMP_OUT_P_BAR"]])
        rows.append(["BOG-RELIQ-RETURN", "Cooled BOG branch returned to LNG line", m_bog_pc, r["BOG_RELIQ_RETURN_T_C"], r["LNG_PUMP_OUT_P_BAR"]])
        rows.append(["LNG-MIX", "Vaporizer feed after reliq branch mixing", r["VAPORIZER_FEED_MASS_FLOW"], r["AFTER_RELIQ_MIX_T_C"], r["VAPORIZER_FEED_P_BAR"]])

    for hx in r["HPC_HX_RESULTS"]:
        rows.append([
            f"FEED-HPC-HX-{hx['hx']}-OUT",
            f"Vaporizer feed outlet after HPC intercooler {hx['hx']}",
            r["VAPORIZER_FEED_MASS_FLOW"],
            K_to_C(hx["COLD_T_out"]),
            Pa_to_bar(hx["COLD_P"]),
        ])

    rows.append(["VAP-IN", "Vaporizer inlet", r["VAPORIZER_FEED_MASS_FLOW"], r["VAPORIZER_FEED_T_C"], r["VAPORIZER_FEED_P_BAR"]])
    rows.append(["VAP-OUT", "Vaporizer outlet before final mixing", r["VAPORIZER_FEED_MASS_FLOW"], r["VAPORIZER_OUT_T_C"], r["MIXED_SENDOUT_P_BAR"]])

    rows.append(["BOG-01", "BOG tank outlet", m_bog_total, r["BOG_TANK_T_C"], r["BOG_TANK_P_BAR"]])
    rows.append(["BOG-02", "BOG compressor outlet before split", m_bog_total, r["BOG_COMP_OUT_T_C"], r["BOG_COMP_OUT_P_BAR"]])

    if r["BOG_PRECOOLER_OUT_T_C"] is not None:
        rows.append(["BOG-PC-IN", "BOG branch to precooler/reliquefier", m_bog_pc, r["BOG_COMP_OUT_T_C"], r["BOG_COMP_OUT_P_BAR"]])
        rows.append(["BOG-PC-OUT", "BOG precooler/reliquefier outlet", m_bog_pc, r["BOG_PRECOOLER_OUT_T_C"], r["BOG_COMP_OUT_P_BAR"]])

    rows.append(["BOG-HPC-IN", "BOG branch to high-pressure compressor", m_bog_hpc, r["BOG_COMP_OUT_T_C"], r["BOG_COMP_OUT_P_BAR"]])

    for st in r["HPC_STAGE_RESULTS"]:
        rows.append([
            f"BOG-HPC-{st['stage']}-OUT",
            f"BOG high-pressure compressor stage {st['stage']} outlet",
            m_bog_hpc,
            K_to_C(st["T_out"]),
            Pa_to_bar(st["P_out"]),
        ])

        for hx in r["HPC_HX_RESULTS"]:
            if hx["after_stage"] == st["stage"]:
                rows.append([
                    f"BOG-HPC-HX-{hx['hx']}-OUT",
                    f"BOG outlet after HPC intercooler {hx['hx']}",
                    m_bog_hpc,
                    K_to_C(hx["BOG_T_out"]),
                    Pa_to_bar(hx["BOG_P"]),
                ])

    rows.append(["BOG-MIX-IN", "BOG before mixer valve", m_bog_hpc, r["BOG_HPC_OUT_T_C"], r["BOG_HPC_OUT_P_BAR"]])
    rows.append(["BOG-MIX", "BOG after valve to mixer", m_bog_hpc, r["BOG_AFTER_VALVE_T_C"], r["BOG_AFTER_VALVE_P_BAR"]])

    rows.append([
        "MIX-01",
        "Mixed sendout gas",
        r["MIXED_SENDOUT_MASS_FLOW"],
        r["MIXED_SENDOUT_T_C"],
        r["MIXED_SENDOUT_P_BAR"],
    ])

    return pd.DataFrame(
        rows,
        columns=["State", "Location", "Mass flow [kg/s]", "Temperature [C]", "Pressure [bar]"],
    )


state_tables = {c: make_state_table(params, c) for c in case_names}

for c in case_names:
    print(f"\n=== STATE TABLE: {c} ===")
    print(state_tables[c].to_string(index=False))


# =====================================================
# 9. Energy summary + export
# =====================================================

energy_cols = [
    "CASE",
    "LNG_PUMP_OUT_HEATING_DUTY_MW",
    "PRECOOLER_DUTY_MW",
    "HPC_INTERCOOLING_DUTY_MW",
    "VAPORIZER_DUTY_MW",
    "LNG_PUMP_POWER_MW",
    "BOG_COMP_POWER_MW",
    "BOG_HPC_POWER_MW",
    "TOTAL_POWER_MW",
    "VAPORIZER_PLUS_POWER_MW",
    "HEATING_PLUS_VAPORIZER_PLUS_POWER_MW",
]

print("\n=== ENERGY SUMMARY ===")
print(df[energy_cols].to_string(index=False))


def export_results_csv(df, summary_cols, energy_cols, state_tables, output_dir="lng_process_results_csv"):
    """Always available export path; does not require openpyxl."""
    os.makedirs(output_dir, exist_ok=True)

    df.drop(columns=["HPC_STAGE_RESULTS", "HPC_HX_RESULTS"]).to_csv(
        os.path.join(output_dir, "case_summary.csv"),
        index=False,
        encoding="utf-8-sig",
    )
    df[summary_cols].to_csv(
        os.path.join(output_dir, "summary.csv"),
        index=False,
        encoding="utf-8-sig",
    )
    df[energy_cols].to_csv(
        os.path.join(output_dir, "energy_summary.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    for case_name, table in state_tables.items():
        table.to_csv(
            os.path.join(output_dir, f"{case_name}.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    return output_dir


def export_results_excel_if_available(df, summary_cols, energy_cols, state_tables, filename="lng_process_results.xlsx"):
    """
    Excel export is optional. If openpyxl is unavailable, skip Excel and keep CSV files.
    """
    try:
        import openpyxl  # noqa: F401
    except ModuleNotFoundError:
        print("\nopenpyxl is not installed. Skipping Excel export and using CSV output instead.")
        return False

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df.drop(columns=["HPC_STAGE_RESULTS", "HPC_HX_RESULTS"]).to_excel(
            writer,
            sheet_name="case_summary",
            index=False,
        )
        df[summary_cols].to_excel(writer, sheet_name="summary", index=False)
        df[energy_cols].to_excel(writer, sheet_name="energy_summary", index=False)

        for case_name, table in state_tables.items():
            table.to_excel(writer, sheet_name=case_name, index=False)

    print(f"\nExcel saved: {filename}")
    return True


csv_dir = export_results_csv(df, summary_cols, energy_cols, state_tables)
print(f"\nCSV files saved in: {csv_dir}")

excel_saved = export_results_excel_if_available(df, summary_cols, energy_cols, state_tables)

if not excel_saved:
    print("Result export completed with CSV files only.")


Smoke tests passed.

=== LNG 기화 공정 3개 Case 비교 ===
                             CASE  BOG_TOTAL_MASS_FLOW  BOG_HPC_MASS_FLOW  BOG_PRECOOLER_MASS_FLOW  LNG_PUMP_OUT_CALC_T_C  LNG_PUMP_OUT_HEATED_T_C  LNG_PUMP_OUT_HEATING_DUTY_MW  LNG_PRECOOLER_OUT_T_C  BOG_RELIQ_RETURN_T_C  AFTER_RELIQ_MIX_T_C  AFTER_HPC_INTERCOOLERS_T_C  VAPORIZER_FEED_MASS_FLOW  VAPORIZER_FEED_T_C  VAPORIZER_OUT_T_C  BOG_COMP_OUT_T_C  BOG_PRECOOLER_OUT_T_C  BOG_HPC_OUT_T_C  BOG_AFTER_VALVE_T_C  MIXED_SENDOUT_T_C  MIXED_SENDOUT_MASS_FLOW  PRECOOLER_DUTY_MW  HPC_INTERCOOLING_DUTY_MW  VAPORIZER_DUTY_MW  BOG_HPC_POWER_MW  TOTAL_POWER_MW  VAPORIZER_PLUS_POWER_MW  HEATING_PLUS_VAPORIZER_PLUS_POWER_MW
                       case1_base                 35.0               21.0                     14.0            -156.074333                   -130.0                      5.957818                    NaN           -124.000000          -128.928346                         NaN                      79.0         -128.928346         -47.1

/usr/local/lib/python3.12/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
